In [3]:
'''
Evaluate the performance of search algorithms 
Collect scores across all prompts and trials, and compute the overall statistics.
'''

import os
import time 
import json
import pprint
import importlib

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import signal

import random
import numpy as np
np.set_printoptions(precision=4)
 
from utils import load_data
# from utils import parser, grader2
from math_verify import parse, verify

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, TimeoutError, as_completed

from sal.config import Config

from tqdm import tqdm



In [9]:
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

def run_with_timeout(completion, gt_answer, timeout=2):
    # Set the signal handler for SIGALRM
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout)  # Schedule an alarm after `timeout` seconds
    try:
        c_answer = parse(completion) 
        # print(c_answer)
        result = verify(gold=gt_answer, target=c_answer)
    except TimeoutException:
        print(f"Timeout: {completion}")
        c_answer = None
        result = None
    finally:
        signal.alarm(0)  # Cancel alarm if function returns early
    return c_answer, result

def compute_samples2correct_onetrial(dataset_orig, result_dir, config_name, trial_idx, config, timeout=2):

    with open(f"{result_dir}/repexp_{config_name}--addprompt-{config.embeds_addprompt}--embstrat-{config.embeds_strategy}--trial-{trial_idx:03d}.jsonl", 'r', encoding='utf-8') as fin:
        gen_results = json.load(fin)

    num_questions = len(dataset_orig)
    # num_questions = 2

    samples2correct = np.zeros(num_questions)
    # for q_idx, data in enumerate(dataset_orig):
    for q_idx in range(num_questions):
        
        # gt_cot, gt_answer = parser.parse_ground_truth(data, 'math')
        gt_answer = parse(dataset_orig[q_idx]['solution'], parsing_timeout=None)
        # print(q_idx, gt_answer)

        q_completions = gen_results["completions"][q_idx]
        q_first_correct_idx = len(q_completions) + 1
        for cidx, completion in enumerate(q_completions):
            # c_answer, is_correct = run_with_timeout(completion, gt_answer)
            ## using math-verify functions parse and verify, if error returns to run_with_timeout function
            c_answer = parse(completion) 
            is_correct = verify(gold=gt_answer, target=c_answer) 
            if is_correct is True: 
                # print(c_answer, is_correct)
                q_first_correct_idx = cidx+1
                break

        # print(q_cnt_corrects)
        samples2correct[q_idx] = q_first_correct_idx
    
    print(samples2correct)
    return samples2correct

    
max_workers = min(16, (os.cpu_count() or 1) * 2)

# base_dir
base_dir = '/groups/chichengz/tnn/datasets/'
# dataset path
data_dir = base_dir + "/prm800k/math_splits"

level = 4
num_trials = 4

config = Config()
config.date_string = "Aug 1 2025"
config.seed = 0

config.lam = 1.0 
config.embeds_centering = True
config.embeds_normalizing = True
config.embeds_strategy = 'avg'
config.embeds_addprompt = 'v02'

config.embeds_dim = 2048

dataset_orig = load_data.load_data_prm800k_hf(data_dir, split='test')
dataset_orig = dataset_orig.filter(lambda example: example['level'] == level)
num_questions = len(dataset_orig)

config_name = f"bon--level-4--v01_0_0--bs-256"
result_dir = f"results/bon--level-{level}/{config_name}"
print(f"config_name = {config_name}")

all_samples2correct = []

for trial_idx in range(num_trials):
    samples2correct = compute_samples2correct_onetrial(dataset_orig, result_dir, config_name, trial_idx, config)
    all_samples2correct.append(samples2correct)

all_samples2correct = np.concatenate(all_samples2correct)

samples2correct_mean = np.mean(all_samples2correct)
samples2correct_std = np.std(all_samples2correct, ddof=1)/np.sqrt(num_trials*num_questions) # 128 is number of prompts for level 4 

print(
    f"{samples2correct_mean:0.4f} (\u00B1{samples2correct_std:0.4f})"
)

config_name = bon--level-4--v01_0_0--bs-256
[ 21.  32.   1.  14. 130.  25.  38.   1.   3. 257.   5.  10.   7. 257.
   1.  15. 257.  21.   1. 110.   7.   1.  30.  36.   1.  36.   1.   1.
  58.  19.  13.   6.  16.  30.  24.  16.   1.   5.  30.  12.  20.  10.
  14.   1.  12.  55.  39.   8.   4.  53. 257.  26.  17. 257.  36.   1.
 104.  13.   1. 126.  15. 257. 125.  25.  13.   1.  46.  20. 257.   4.
   4.  44.  17. 106. 257.  71. 257. 257.   1. 202.  21.   1.  21.   1.
  27.   4.  52.   1.  13.  83.   1.   1.   1.  22. 257.  20.  24.  15.
 257. 257.  10.   3.   7.  12.  16.   1.   1.   4.   1.   1.  68.   4.
   1. 257.  50. 257.  26.   1.   1.   6.  42.  33.   8.   1.   5.   3.
 257.  14.]
[  7.  13.   1.   1.  13.  63.  21.   1.   4. 257.   7.  15.  73. 257.
   1.   9. 257.  16.  21. 129.  11.   1.   4.  46.  11.  68.   1.   1.
  33.  10.   1.  30.   6.  10.  37.   5.   1.  12.  33.  14.  70.   4.
  13.   8.  26.   1.   3.  17.  18.  16. 257.  34.   1. 257.  22.   2.
 226.   1.   1.   7. 